# Significance testing

Paired one-sided t-tests across the six walk-forward test years (PPO, DQN,
MktCap_Weight vs Equal_Weight) and a paired two-sided test for PPO vs DQN.

**Input:** `results/pipeline_b_test_results.csv` (six windows × five algorithms, one row each).

**Output:** `results/significance_tests.csv` (four rows, one per paired test).

This notebook does **not** modify `results/pipeline_b_summary.xlsx` — that
file's `ttests` sheet is owned by Notebooks 5 and 6. The CSV produced here is
a standalone canonical record of the significance results reported in §5.2.2
of the thesis.

Pipeline A is intentionally omitted. The baselines (Equal_Weight, MktCap_Weight,
Buy_Hold) appear exactly once in `pipeline_a_test_results.csv` with no seed
dimension, so a paired test against EW would collapse to a one-sample
comparison against a constant on n = 3 — a degenerate-power test that is not
informative. Pipeline A test results are reported in §5.1.2 of the thesis.


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path


In [2]:
RESULTS = Path("results")
INPUT_CSV = RESULTS / "pipeline_b_test_results.csv"
OUTPUT_CSV = RESULTS / "significance_tests.csv"

# (pipeline, comparison_label, algo_a, algo_b, tail)
# H1 for one-sided tests: algo_a > algo_b.
COMPARISONS = [
    ("B", "PPO vs Equal_Weight",           "PPO",           "Equal_Weight", "greater"),
    ("B", "DQN vs Equal_Weight",           "DQN",           "Equal_Weight", "greater"),
    ("B", "MktCap_Weight vs Equal_Weight", "MktCap_Weight", "Equal_Weight", "greater"),
    ("B", "PPO vs DQN",                    "PPO",           "DQN",          "two_sided"),
]


In [3]:
wf = pd.read_csv(INPUT_CSV)
wide = (
    wf.pivot_table(index="window", columns="algorithm", values="sharpe")
      .sort_index()
)
wide = wide[["Buy_Hold", "Equal_Weight", "MktCap_Weight", "PPO", "DQN"]]

assert wide.shape == (6, 5), f"expected 6 windows x 5 algorithms, got {wide.shape}"

print(f"Loaded {INPUT_CSV.name}: {len(wf)} rows")
print(f"Pivoted to wide form: {wide.shape[0]} windows x {wide.shape[1]} algorithms")
print()
print("Per-window Sharpe (Pipeline B test):")
print(wide.round(4).to_string())


Loaded pipeline_b_test_results.csv: 30 rows
Pivoted to wide form: 6 windows x 5 algorithms

Per-window Sharpe (Pipeline B test):
algorithm  Buy_Hold  Equal_Weight  MktCap_Weight     PPO     DQN
window                                                          
1           -0.0535       -0.0632         0.2418 -0.2427 -0.3342
2            2.5478        2.5851         3.2971  2.5372  2.5684
3            0.8646        0.9206         1.2985  0.7329  0.8046
4            1.5627        1.6701         2.3331  1.8049  1.6982
5           -0.8062       -0.7470        -0.7023 -0.8153 -0.7282
6            2.1356        2.2314         3.2735  2.2596  2.4977


In [4]:
def paired_ttest(a, b):
    """
    Paired t-test on two equal-length 1-D arrays.

    Returns a dict with: t, df, p_two_sided, p_one_sided_greater, mean_diff, n.

    One-sided p (H1: a > b) follows the NB5/NB6 idiom:
        p_one = p_two / 2  if t > 0  else  1 - p_two / 2
    which is identical to verify_thesis_numbers.py for any non-zero t.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    assert a.shape == b.shape and a.ndim == 1, "a and b must be 1-D arrays of equal length"
    n = int(a.shape[0])
    t_stat, p_two = stats.ttest_rel(a, b)
    p_one_greater = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    return {
        "t": float(t_stat),
        "df": n - 1,
        "p_two_sided": float(p_two),
        "p_one_sided_greater": float(p_one_greater),
        "mean_diff": float((a - b).mean()),
        "n": n,
    }


In [5]:
rows = []
for pipeline, comparison, algo_a, algo_b, tail in COMPARISONS:
    a = wide[algo_a].values
    b = wide[algo_b].values
    res = paired_ttest(a, b)
    rows.append({
        "pipeline":    pipeline,
        "comparison":  comparison,
        "algo_a":      algo_a,
        "algo_b":      algo_b,
        "mean_diff":   res["mean_diff"],
        "t_stat":      res["t"],
        "df":          res["df"],
        "p_two_sided": res["p_two_sided"],
        "p_one_sided": res["p_one_sided_greater"] if tail == "greater" else np.nan,
        "n":           res["n"],
        "tail":        tail,
    })

tests_df = pd.DataFrame(rows, columns=[
    "pipeline", "comparison", "algo_a", "algo_b",
    "mean_diff", "t_stat", "df", "p_two_sided", "p_one_sided", "n", "tail",
])
tests_df


,pipeline,comparison,algo_a,algo_b,mean_diff,t_stat,df,p_two_sided,p_one_sided,n,tail
0,B,PPO vs Equal_Weight,PPO,Equal_Weight,-0.053400,-1.059086,5,0.338019,0.830991,6,greater
1,B,DQN vs Equal_Weight,DQN,Equal_Weight,-0.015083,-0.207926,5,0.843491,0.578255,6,greater
2,B,MktCap_Weight vs Equal_Weight,MktCap_Weight,Equal_Weight,0.524117,3.639561,5,0.014910,0.007455,6,greater
3,B,PPO vs DQN,PPO,DQN,-0.038317,-0.735860,5,0.494866,NaN,6,two_sided


In [6]:
tests_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV} ({len(tests_df)} rows)")


Wrote results/significance_tests.csv (4 rows)


In [7]:
# Thesis-ready summary table (display rounding only; CSV stores full precision)
print()
print("=" * 78)
print("Pipeline B paired t-tests (n = 6 walk-forward windows)")
print("=" * 78)
header = f"{'Comparison':<35} {'mean_diff':>10} {'t':>8} {'p_two':>9} {'p_one':>9}"
print(header)
print("-" * len(header))
for r in rows:
    p_one_str = f"{r['p_one_sided']:.4f}" if not np.isnan(r['p_one_sided']) else "    --   "
    print(
        f"{r['comparison']:<35} "
        f"{r['mean_diff']:>10.4f} "
        f"{r['t_stat']:>8.3f} "
        f"{r['p_two_sided']:>9.4f} "
        f"{p_one_str:>9}"
    )
print()
print("Note: 'p_one' is the one-sided p-value testing H1: algo_a > algo_b.")
print("For 'PPO vs DQN' the test is two-sided; p_one is not reported.")
print()

print("=" * 78)
print("Conclusion")
print("=" * 78)
print(
    "Neither PPO nor DQN significantly beats the Equal-Weight baseline on the\n"
    "Pipeline B walk-forward (one-sided p = 0.831 for PPO, 0.578 for DQN). Only\n"
    "Market-Cap Weight significantly outperforms Equal-Weight (one-sided p = 0.0075).\n"
    "PPO and DQN are not significantly different from each other (two-sided p = 0.495).\n"
    "\n"
    "The DRL agents do not statistically outperform a naive equal-weight baseline on\n"
    "this 47-stock IT universe; the market-cap-weighted baseline does, indicating that\n"
    "the universe's large-cap bias - not learned policy - is what is producing the\n"
    "only significant result."
)



Pipeline B paired t-tests (n = 6 walk-forward windows)
Comparison                           mean_diff        t     p_two     p_one
---------------------------------------------------------------------------
PPO vs Equal_Weight                    -0.0534   -1.059    0.3380    0.8310
DQN vs Equal_Weight                    -0.0151   -0.208    0.8435    0.5783
MktCap_Weight vs Equal_Weight           0.5241    3.640    0.0149    0.0075
PPO vs DQN                             -0.0383   -0.736    0.4949     --   

Note: 'p_one' is the one-sided p-value testing H1: algo_a > algo_b.
For 'PPO vs DQN' the test is two-sided; p_one is not reported.

Conclusion
Neither PPO nor DQN significantly beats the Equal-Weight baseline on the
Pipeline B walk-forward (one-sided p = 0.831 for PPO, 0.578 for DQN). Only
Market-Cap Weight significantly outperforms Equal-Weight (one-sided p = 0.0075).
PPO and DQN are not significantly different from each other (two-sided p = 0.495).

The DRL agents do not statistic

## Conclusion

Neither PPO nor DQN significantly beats the Equal-Weight baseline on the
Pipeline B walk-forward (one-sided p = 0.831 for PPO, 0.578 for DQN). Only
Market-Cap Weight significantly outperforms Equal-Weight (one-sided p = 0.0075).
PPO and DQN are not significantly different from each other (two-sided p = 0.495).

The DRL agents do not statistically outperform a naïve equal-weight baseline
on this 47-stock IT universe; the market-cap-weighted baseline does, indicating
that the universe's large-cap bias — not learned policy — is what is
producing the only significant result.
